In [1]:
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd

In [ ]:
dataset = ['IRIS','DERMATOLOGY','WINE','MUSK','OPTDIGITS389','BreastCancerWisconsin','PARKINSONS','ECOLI','SEGMENTATION','IONOSPHERE', 'HCV', 'SPAMBASE','SONAR','GLASS','COLUMN_2C','YEAST','PIMA','SEEDS','TAE',
            'HABERMAN','SOYBEAN','BALANCE','VEHICLE','BANKNOTE','WAVEFORM','EEG','LETTER','mfeat_zernike','mfeat_karhunen','SKIN','MNIST']

from FALCON.open_data import open_dataset
import warnings
warnings.filterwarnings("ignore")

dic_X = {}
dic_y = {}
for name in dataset :
    X,y = open_dataset(name)
    dic_X[name] = X
    dic_y[name] = y

In [ ]:
### This correspond to the best sigma value found by GridSearch analysis

dic_sigma = {'IRIS': (2,0.5),
 'DERMATOLOGY': (1,0.025),
 'WINE': (0.1,0.5),
 'MUSK': (1,1),
 'OPTDIGITS389': (1,0.025),
 'BreastCancerWisconsin': (0.25,0.25),
 'PARKINSONS': (0.1,0.025),
 'ECOLI': (10,0.25),
 'SEGMENTATION': (0.25,0.025),
 'IONOSPHERE': (0.5,10),
 'HCV': (0.25,0.5),
 'SPAMBASE': (0.5,0.025),
 'SONAR': (0.25,0.5),
 'GLASS': (0.25,0.1),
 'COLUMN_2C': (0.5,0.025),
 'YEAST': (0.1,2),
 'PIMA': (0.1,1),
 'SEEDS': (0.25,10),
 'TAE': (0.5,10),
 'HABERMAN': (0.1,0.1),
 'SOYBEAN': (0.5,0.1),
 'BALANCE': (0.1,10),
 'VEHICLE': (0.25,0.25),
 'BANKNOTE': (0.1,10),
 'WAVEFORM': (1,0.25),
 'EEG': (0.25,0.1),
 'LETTER': (2,0.25),
 'mfeat_zernike': (0.5,0.5),
 'mfeat_karhunen': (2,2),
 'SKIN' : (0.1,0.1),
 'MNIST': (2,10)}

In [ ]:
y_proto = np.array([dic_sigma[name][0] for name in dataset]) 
y_criti = np.array([dic_sigma[name][1] for name in dataset]) 

from pymfe.mfe import MFE
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

dic_mfe = {}
for name in tqdm(dataset):
    mfe = MFE(features = ['sil','vdb','vdu','c1','c2','cls_coef','complexity','density','inst_to_attr','freq_class', 'nr_attr','nr_bin','nr_cat','nr_class','nr_inst','nr_num', 'num_to_cat','cov','cor','eigenvalues','g_mean','mean','median','min','mad','sd','var'])
    mfe.fit(dic_X[name])
    ft = mfe.extract()
    dic_mfe[name] = ft[1]

In [ ]:
df = pd.DataFrame.from_dict(dic_mfe, orient='index', columns = ft[0])
df = df.dropna(axis=1)
df = df.drop_duplicates()

In [ ]:
from sklearn import tree
from sklearn.preprocessing import StandardScaler

X = np.array(df)

clf = tree.DecisionTreeClassifier(max_depth = 3)
clf = clf.fit(X, y_proto.astype(str))
y_pred_proto = clf.predict(X)

In [ ]:
import matplotlib
import re

fig, ax = plt.subplots(figsize=(18,10))
tree.plot_tree(clf, feature_names= df.columns, class_names = y_pred_proto,proportion=False,ax=ax)

def replace_text(obj):
    if type(obj) == matplotlib.text.Annotation:
        txt = obj.get_text()
        txt = re.sub("\nvalue[^$]*class","\nclass",txt)
        obj.set_text(txt)
    return obj
    
ax.properties()['children'] = [replace_text(i) for i in ax.properties()['children']]
fig.show()

plt.savefig('/projects/sig/vblase/Resultats_expe/tree1.png', format='png', dpi=300)

In [ ]:
X_2 = np.c_[X, y_pred_proto]

clf = tree.DecisionTreeClassifier(max_depth = 3)
clf = clf.fit(X_2, y_criti.astype(str))

y_pred_criti = clf.predict(X_2)

In [ ]:
fig, ax = plt.subplots(figsize=(18,10))
tree.plot_tree(clf, feature_names= df.columns, class_names = y_pred_proto,proportion=False,ax=ax)

def replace_text(obj):
    if type(obj) == matplotlib.text.Annotation:
        txt = obj.get_text()
        txt = re.sub("\nvalue[^$]*class","\nclass",txt)
        obj.set_text(txt)
    return obj
    
ax.properties()['children'] = [replace_text(i) for i in ax.properties()['children']]
fig.show()

plt.savefig('/projects/sig/vblase/Resultats_expe/tree2.png', format='png', dpi=300)